# Grouped vs Pooled — Cleaned Version

This notebook is an extracted, commented version of your original workflow. 
Two variants are provided:
1) **SSE-only** — no robust losses, no calculated means in summaries.
2) **Robust-only (Huber + Soft‑L1)** — uses Huber (IRLS) in the offset–power model and Soft‑L1 for the linear fit; no calculated means.

Notes:
- Both variants load `pooled_event_df.csv` (or `../data/processed/pooled_event_df.csv` as a fallback).
- Both variants provide: pooled evaluation by thresholds and per‑event fits.
- “No calculated means” = summary tables avoid computing/printing means; medians are retained where useful.


In [ ]:

# --- Imports & config (Robust-only: Huber + Soft-L1) ---
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares, minimize_scalar

THRESHOLDS = [5,6,7,8,10,12,15]
TEST_FRAC  = 0.20
SEED       = 42
EPS        = 1e-12

# Robust params
HUBER_C = 1.345  # ~95% efficiency for Gaussian errors

DATA_PATHS = [Path('pooled_event_df.csv'), Path('../data/processed/pooled_event_df.csv')]
DATA_PATH = next((p for p in DATA_PATHS if p.exists()), DATA_PATHS[-1])
print('Using data:', DATA_PATH)


In [ ]:

# --- Load + basic cleaning ---
df = pd.read_csv(DATA_PATH)
for col in ["delta_h", "seepage"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
if "visitor_h" not in df.columns:
    df["visitor_h"] = np.nan
else:
    df["visitor_h"] = pd.to_numeric(df["visitor_h"], errors="coerce")

df = df.dropna(subset=["seepage"]).copy()
df["n_full"] = df.groupby("event")["event"].transform("size").astype("int64")
df = df.query('n_full > 5 and seepage < 4')
print(df[["event","delta_h","visitor_h","seepage","n_full"]].head())


In [ ]:

# --- Models & robust utilities ---

def model_linear(K, x):
    return K * x

def model_power_offset_normref(K, off, b, x, x_ref):
    x_ref = max(float(x_ref), EPS)
    xp = np.maximum(x + off, EPS)
    return K * x * np.power(xp / x_ref, b)

def r2_centered(y, yhat):
    y = np.asarray(y, float); yhat = np.asarray(yhat, float)
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - np.mean(y))**2) + EPS
    return 1.0 - ss_res/ss_tot

def rmse(y, yhat):
    y = np.asarray(y, float); yhat = np.asarray(yhat, float)
    return float(np.sqrt(np.mean((y - yhat)**2)))

def mad(arr):
    arr = np.asarray(arr, float)
    med = np.median(arr)
    return np.median(np.abs(arr - med))

def huber_rho_sum(r, delta):
    # Sum of Huber's rho for residual vector r with threshold delta.
    a = np.abs(r)
    quad = 0.5 * (a**2)
    lin  = delta * (a - 0.5*delta)
    return float(np.sum(np.where(a <= delta, quad, lin)))

def robust_wls_log_reg(Z, t, huber_delta=None, max_iter=50, tol=1e-8):
    """
    IRLS for robust log-linear regression:
        t ≈ alpha + b * Z
    Returns (alpha, b). Huber weights with threshold = huber_delta or HUBER_C * (1.4826*MAD(resid)).
    """
    Z = np.asarray(Z, float); t = np.asarray(t, float)
    X = np.column_stack([np.ones(len(Z)), Z])
    # init by OLS
    beta = np.linalg.lstsq(X, t, rcond=None)[0]
    for _ in range(max_iter):
        r = t - X @ beta
        s = 1.4826 * mad(r) + 1e-9
        delta = (HUBER_C * s) if huber_delta is None else huber_delta
        a = np.abs(r)
        w = np.where(a <= delta, 1.0, delta / (a + 1e-12))
        Xw = X * np.sqrt(w)[:, None]
        tw = t * np.sqrt(w)
        beta_new = np.linalg.lstsq(Xw, tw, rcond=None)[0]
        if np.max(np.abs(beta_new - beta)) < tol * (np.abs(beta).sum() + tol):
            beta = beta_new
            break
        beta = beta_new
    alpha, b = beta
    return float(alpha), float(b)

def aic_gaussian_from_resid(resid, k):
    r = np.asarray(resid, float)
    n = len(r)
    sse = float(np.dot(r, r))
    aic = n * np.log(sse/max(n,1)) + 2*int(k)
    # Finite-sample correction
    if n - k - 1 <= 0: 
        return aic
    return aic + (2*k*(k+1)) / (n - k - 1)


In [ ]:

# --- Linear-through-origin (Soft-L1 via least_squares) ---

def fit_linear_softl1(x, y):
    """
    Fit y ≈ K * x using scipy.least_squares with loss='soft_l1'.
    Scale f_scale from robust MAD of initial residuals.
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    # init K by closed form
    K0 = float(np.dot(x, y) / (np.dot(x, x) + EPS))
    def resid(p):  # vector residuals
        return y - (p[0] * x)
    r0 = resid([K0])
    f_scale = 1.4826 * mad(r0) + 1e-9
    ls = least_squares(lambda p: resid(p), x0=[max(K0, EPS)],
                       bounds=([EPS],[np.inf]), loss="soft_l1", f_scale=f_scale)
    K = float(ls.x[0])
    return K


In [ ]:

# --- Offset–power robust fit: Huber objective over offset + IRLS in log space ---

def solve_kb_for_offset_normref_robust(off_bounds, x, y, x_ref):
    x = np.asarray(x, float); y = np.asarray(y, float); x_ref = float(x_ref)
    lb, ub = off_bounds

    def fit_given_off(off):
        xp = np.maximum(x + off, EPS)
        Z = np.log(xp / x_ref)
        t = np.log(np.maximum(y, EPS)) - np.log(np.maximum(x, EPS))
        alpha, b = robust_wls_log_reg(Z, t)
        K = float(np.exp(alpha))
        yhat = model_power_offset_normref(K, off, b, x, x_ref)
        return K, b, yhat

    # scalar robust objective over offset using Huber rho on original-space residuals
    def obj(off):
        K, b, yhat = fit_given_off(off)
        r = y - yhat
        s = 1.4826 * mad(r) + 1e-9
        delta = HUBER_C * s
        return huber_rho_sum(r, delta)

    # coarse-to-fine search for offset
    grid = np.linspace(lb, ub, 21)
    best_off = grid[0]; best_val = np.inf
    for off in grid:
        val = obj(off)
        if val < best_val:
            best_val, best_off = val, off
    w = max( (ub - lb) * 0.1, 1e-3 )
    lb2, ub2 = max(lb, best_off - w), min(ub, best_off + w)
    res = minimize_scalar(obj, bounds=(lb2, ub2), method="bounded")
    off_star = float(res.x)

    K, b, yhat = fit_given_off(off_star)
    r = y - yhat
    aic_pow = aic_gaussian_from_resid(r, k=3)  # approximate AIC on residuals
    return K, off_star, b, yhat, aic_pow


In [ ]:

# --- Splits + bounds ---

def make_fixed_cohort_and_holdout(df_in, thresholds, x_col):
    NSTAR = max(thresholds)
    dfx = df_in.dropna(subset=[x_col, "seepage"]).copy()
    dfx["n"] = dfx.groupby("event")["event"].transform("size").astype("int64")
    cohort = dfx.loc[dfx["n"] >= NSTAR].copy()
    if cohort.empty:
        NSTAR = int(dfx["n"].max())
        cohort = dfx.loc[dfx["n"] >= NSTAR].copy()

    parts_pool, parts_hold = [], []
    for e, g in cohort.groupby("event"):
        seed_for_event = SEED + (abs(hash(str(e))) % 10**6)
        g = g.sample(frac=1.0, random_state=seed_for_event).reset_index(drop=True)
        n_e = len(g); n_test = max(1, int(np.ceil(TEST_FRAC * n_e)))
        parts_hold.append(g.iloc[:n_test].assign(split="test"))
        parts_pool.append(g.iloc[n_test:].assign(split="train"))
    return pd.concat(parts_pool, ignore_index=True), pd.concat(parts_hold, ignore_index=True)

def global_offset_bounds(x_series):
    x = x_series.to_numpy(float)
    offset_lb = -float(np.nanmin(x)) + 1e-9
    q05, q95 = np.nanquantile(x, [0.05, 0.95])
    R = float(q95 - q05)
    site_cap = 1.0
    offset_ub = min(0.30*R, site_cap)
    if offset_ub <= offset_lb + 1e-6:
        offset_ub = offset_lb + max(0.05*R, 0.05)
    return offset_lb, offset_ub


In [ ]:

# --- Pooled evaluation by thresholds (Robust-only) ---

def eval_by_thresholds_robust(train_df, hold_df, thresholds, x_col):
    rows = []
    for n in thresholds:
        def last_n(g): 
            g = g.sort_index()
            return g.tail(n)
        train_n = (train_df.groupby("event", group_keys=False).apply(last_n)
                   .dropna(subset=[x_col, "seepage"])
                   .reset_index(drop=True))

        x_ref = float(np.median(train_n[x_col].to_numpy(float)))
        x = train_n[x_col].to_numpy(float); y = train_n["seepage"].to_numpy(float)

        # Linear (Soft-L1)
        K_lin = fit_linear_softl1(x, y)
        yhat_lin = model_linear(K_lin, x)
        aic_lin = aic_gaussian_from_resid(y - yhat_lin, k=1)
        rmse_lin = rmse(y, yhat_lin)

        # Offset–power (Huber + IRLS)
        lb, ub = global_offset_bounds(train_n[x_col])
        K_pow, off_star, b_pow, yhat_pow, aic_pow = solve_kb_for_offset_normref_robust((lb, ub), x, y, x_ref)
        rmse_pow = rmse(y, yhat_pow)

        # Holdout
        te = hold_df.loc[hold_df["event"].isin(train_n["event"].unique())]
        x_te = te[x_col].to_numpy(float); y_te = te["seepage"].to_numpy(float)

        yhat_te_lin = model_linear(K_lin, x_te)
        yhat_te_pow = model_power_offset_normref(K_pow, off_star, b_pow, x_te, x_ref)
        r2_te_lin = r2_centered(y_te, yhat_te_lin)
        r2_te_pow = r2_centered(y_te, yhat_te_pow)

        rows.append({
            "n": n, "n_train": int(len(train_n)), "x_col": x_col, "x_ref": x_ref,
            "AICc_lin": aic_lin, "AICc_pow": aic_pow,
            "RMSE_lin": rmse_lin, "RMSE_pow": rmse_pow, "dRMSE": rmse_pow - rmse_lin,
            "R2_te_lin": r2_te_lin, "R2_te_pow": r2_te_pow,
            "K_lin": K_lin, "K_pow": K_pow, "offset": off_star, "b": b_pow
        })
    out = pd.DataFrame(rows)
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(3)
    out[num_cols] = out[num_cols].mask(np.isclose(out[num_cols], 0, atol=5e-4), 0.0)
    return out


In [ ]:

# --- Per-event fits (Robust-only) ---

def per_event_fits_robust(df_in, x_col, min_n=6):
    dfx = df_in.dropna(subset=[x_col, "seepage"]).copy()
    rows = []
    for e, g in dfx.groupby("event"):
        x = g[x_col].to_numpy(float); y = g["seepage"].to_numpy(float)
        n = len(x)
        if n < min_n: 
            continue
        x_ref_evt = float(np.median(x))
        lb, ub = global_offset_bounds(g[x_col])

        # Linear (Soft-L1)
        K_lin = fit_linear_softl1(x, y)
        yhat_l = model_linear(K_lin, x)
        r2l = r2_centered(y, yhat_l)

        # Offset–power (Huber + IRLS)
        K_pow, offp, bp, yhat_p, _ = solve_kb_for_offset_normref_robust((lb, ub), x, y, x_ref_evt)
        r2p = r2_centered(y, yhat_p)

        rows.append({
            "event": e, "n": n, "x_col": x_col, "x_ref_evt": x_ref_evt,
            "K_lin": K_lin, "R2_lin": r2l,
            "K_pow": K_pow, "offset": offp, "b": bp, "R2_pow": r2p
        })
    out = pd.DataFrame(rows).sort_values("event").reset_index(drop=True)
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(3)
    out[num_cols] = out[num_cols].mask(np.isclose(out[num_cols], 0, atol=5e-4), 0.0)
    return out


In [ ]:

# --- Run pooled evals + per-event fits (Robust-only) ---
train_pool_dh, holdout_dh = make_fixed_cohort_and_holdout(df, THRESHOLDS, "delta_h")
eval_dh = eval_by_thresholds_robust(train_pool_dh, holdout_dh, THRESHOLDS, "delta_h")

has_vis = df["visitor_h"].notna().any()
if has_vis:
    df_vis = df.dropna(subset=["visitor_h"]).copy()
    train_pool_vis, holdout_vis = make_fixed_cohort_and_holdout(df_vis, THRESHOLDS, "visitor_h")
    eval_vis = eval_by_thresholds_robust(train_pool_vis, holdout_vis, THRESHOLDS, "visitor_h")
else:
    eval_vis = pd.DataFrame()

def best_row_by_rmse(df_eval):
    i_lin = df_eval["RMSE_lin"].astype(float).idxmin()
    i_pow = df_eval["RMSE_pow"].astype(float).idxmin()
    return df_eval.loc[i_lin], df_eval.loc[i_pow]

rows = []
best_lin_dh, best_pow_dh = best_row_by_rmse(eval_dh)
rows += [{
    "model": "Linear (delta_h)", "n_best": int(best_lin_dh["n"]),
    "RMSE_test": float(best_lin_dh["RMSE_lin"]), "R2_test": float(best_lin_dh["R2_te_lin"]),
    "AICc_train": float(best_lin_dh["AICc_lin"]), "K": float(best_lin_dh["K_lin"])
}, {
    "model": "Offset–power (delta_h)", "n_best": int(best_pow_dh["n"]),
    "RMSE_test": float(best_pow_dh["RMSE_pow"]), "R2_test": float(best_pow_dh["R2_te_pow"]),
    "AICc_train": float(best_pow_dh["AICc_pow"]), "K": float(best_pow_dh["K_pow"]),
    "offset": float(best_pow_dh["offset"]), "b": float(best_pow_dh["b"]),
}]
if has_vis and not eval_vis.empty:
    _, best_pow_vis = best_row_by_rmse(eval_vis)
    rows += [{
        "model": "Offset–power (visitor_h)",
        "n_best": int(best_pow_vis["n"]), "RMSE_test": float(best_pow_vis["RMSE_pow"]),
        "R2_test": float(best_pow_vis["R2_te_pow"]), "AICc_train": float(best_pow_vis["AICc_pow"]),
        "K": float(best_pow_vis["K_pow"]), "offset": float(best_pow_vis["offset"]), "b": float(best_pow_vis["b"]),
    }]

pooled_best = pd.DataFrame(rows)
display(pooled_best)

ev_dh = per_event_fits_robust(df, "delta_h", min_n=6)
ev_vs = per_event_fits_robust(df, "visitor_h", min_n=5) if has_vis else pd.DataFrame()

# Save
pooled_best.to_csv("pooled_best_robust.csv", index=False)
ev_dh.to_csv("per_event_delta_h_robust.csv", index=False)
if not ev_vs.empty:
    ev_vs.to_csv("per_event_visitor_h_robust.csv", index=False)
print("Saved CSVs next to this notebook.")
